In [0]:
dbutils.widgets.help()

dbutils.widgets provides utilities for working with notebook widgets. You can create
different types of widgets and get their bound value.

For more info about a method, use dbutils.widgets.help("methodName") .
 combobox(name: String, defaultValue: String, choices: Seq, label: String): void -> Creates a combobox input widget with a given name, default value and choices dropdown(name: String, defaultValue: String, choices: Seq, label: String): void -> Creates a dropdown input widget a with given name, default value and choices get(name: String): String -> Retrieves current value of an input widget getAll: Map -> Retrieves a mapping of all current values of the input widgets getArgument(name: String, optional: String): String -> (DEPRECATED) Equivalent to get multiselect(name: String, defaultValue: String, choices: Seq, label: String): void -> Creates a multiselect input widget with a given name, default value and choices remove(name: String): void -> Removes an input widget from the notebook removeAll: void -> Removes all widgets in the notebook text(name: String, defaultValue: String, label: String): void -> Creates a text input widget with a given name and default value

In [0]:
dbutils.widgets.text("p_environment", "")
v_environment = dbutils.widgets.get("p_environment")

In [0]:
dbutils.widgets.text("p_file_date", "2024-12-30")
v_file_date = dbutils.widgets.get("p_file_date")

In [0]:
%run "../includes/configuration"

In [0]:
%run "../includes/commom_functions"

## Ingestion del archivo "movie.csv"

###Paso 1 - Leer el archivo CSV usando "DataframeReader" de Spark

In [0]:
from pyspark.sql.types import StructType, StructField, IntegerType, StringType, DoubleType, DateType

In [0]:
movie_schema = StructType( fields= [
    StructField("movieId", IntegerType(), False),
    StructField("title", StringType(), True),
    StructField("budget", DoubleType(), True),
    StructField("homepage", StringType(), True),
    StructField("overview", StringType(), True),
    StructField("popularity", DoubleType(), True),
    StructField("yearReleaseDate", IntegerType(), True),
    StructField("releaseDate", DateType(), True),
    StructField("revenue", DoubleType(), True),
    StructField("durationTime", IntegerType(), True),
    StructField("movieStatus", StringType(), True),
    StructField("tagline", StringType(), True),
    StructField("voteAverage", DoubleType(), True),
    StructField("voteCount", IntegerType(), True)
])

In [0]:
movie_df = spark.read \
    .option("header", True) \
    .schema(movie_schema) \
    .csv(f"{bronze_folder_path}/{v_file_date}/movie.csv")

### Paso 2 - Seleccionar sólo las columnas "requeridas"

In [0]:
# movies_selected_df = movie_df.select("movieId", "title", "budget", "popularity", "yearReleaseDate", "releaseDate", "revenue", "durationTime", "voteAverage", "voteCount")

In [0]:
from pyspark.sql.functions import col

In [0]:
movies_selected_df = movie_df.select(col("movieId"), col("title"), col("budget"), col("popularity"), col("yearReleaseDate"), col("releaseDate"), col("revenue"), col("durationTime"), col("voteAverage"), col("voteCount"))

### Paso 3 - Cambiar el nombre de las columnas según lo "requerido"

In [0]:
movies_rename_df = movies_selected_df \
    .withColumnRenamed("movieId", "movie_id") \
    .withColumnRenamed("yearReleaseDate", "year_release_date") \
    .withColumnRenamed("releaseDate", "release_date") \
    .withColumnRenamed("durationTime", "duration_time") \
    .withColumnRenamed("voteAverage", "vote_average") \
    .withColumnRenamed("voteCount", "vote_count")

In [0]:
# movies_rename_df = movies_selected_df \
#     .withColumnRenamed({
#         "movieId": "movie_id", 
#         "yearReleaseDate": "year_release_date", 
#         "releaseDate": "release_date", 
#         "durationTime": "duration_time",
#         "voteAverage": "vote_average", 
#         "voteCount": "vote_count"
#         })

### Paso 4 - Agregar la columna "ingestion_date" al DataFrame

In [0]:
from pyspark.sql.functions import current_timestamp, lit

In [0]:
movies_final_df = add_ingestion_date(movies_rename_df)\
                    .withColumn("environment", lit(v_environment))\
                    .withColumn("file_date", lit(v_file_date))

In [0]:
# movies_final_df = movies_rename_df.withColumns({"ingestion_date": current_timestamp(), "env": lit("production")})

### Paso 5 - Escribir datos en el datalake en formato "Parquet"

In [0]:
#overwrite_partition("movie_silver", "movies", "file_date", v_file_date)

In [0]:
# from delta.tables import DeltaTable

# if spark.catalog.tableExists("movie_silver.movies"):

#     deltaTable = DeltaTable.forName(spark, 'movie_silver.movies')

#     deltaTable.alias('tgt') \
#     .merge(
#         movies_final_df.alias('src'),
#         'tgt.movie_id = src.movie_id AND tgt.file_date = src.file_date'
#     ) \
#     .whenMatchedUpdateAll() \
#     .whenNotMatchedInsertAll() \
#     .execute()

# else:

#     movies_final_df.write.mode("overwrite").partitionBy("file_date").format("delta").saveAsTable("movie_silver.movies")
condition_merge = 'tgt.movie_id = src.movie_id AND tgt.file_date = src.file_date'

incremental_merge("movie_silver", "movies", movies_final_df, condition_merge, "file_date")

In [0]:
%sql
SELECT file_date, count(1)
FROM movie_silver.movies 
GROUP BY file_date;

In [0]:
dbutils.notebook.exit("Exitoso")